# 04 Prompt Contracts and Structured Output (OpenClaw, 2026)

## What This Lesson Is
Use explicit output contracts and parsers so OpenClaw responses are testable and automatable.

## Scientific Lens
- Concept: Structured output contract enforcement with parser validation.
- Measure: Parse success rate and schema conformance.
- Validity Limit: Prompt-only constraints cannot guarantee strict JSON without retries/guards.


## How It Works
1. Define deterministic schema and validate sample outputs.
2. Reject malformed payloads before downstream usage.
3. Run live OpenClaw query requesting JSON and validate returned shape.


In [ ]:
import os
import shutil

HAS_OPENCLAW = shutil.which("openclaw") is not None
print("openclaw available:", HAS_OPENCLAW)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("OLLAMA_BASE_URL:", os.getenv("OLLAMA_BASE_URL", "<unset>"))


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: runs real OpenClaw CLI operations when available; otherwise prints explicit skip guidance.


In [ ]:
# Deterministic Demo
import json
required_keys = {"summary", "risk", "next_step"}
samples = ['{"summary":"ok","risk":"low","next_step":"deploy"}', '{"summary":"missing fields"}']
results = []
for raw in samples:
    data = json.loads(raw)
    results.append(required_keys.issubset(data.keys()))
print("schema_validity:", results)
assert results == [True, False]


In [ ]:
# Live Demo
import json
import shutil
import subprocess

if shutil.which("openclaw") is None:
    print("Skipping live structured-output demo: openclaw CLI is not installed.")
else:
    prompt = "Return only JSON with keys summary, risk, next_step. Topic: why guardrails matter in AI agents."
    cmd = ["openclaw", "agent", "--local", "--to", "+15555550123", "--message", prompt, "--timeout", "180"]
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    out = (proc.stdout or proc.stderr).strip()
    print(out[:1200])
    try:
        parsed = json.loads(out[out.find('{'): out.rfind('}') + 1])
        print("parsed keys:", sorted(parsed.keys()))
    except Exception as exc:
        print(f"Could not parse JSON payload: {exc}")


## Applied Labs
1. Add numeric confidence score and enforce `0.0 <= score <= 1.0`.
2. Implement retry wrapper requesting corrected JSON when parse fails.
3. Create regression fixtures with valid/invalid outputs and run parser tests.

## Validation Checklist
- Schema validator checks required keys before accepting output.
- Malformed JSON paths are handled without uncaught exceptions.
- Live demo logs parse outcome and avoids silent failures.

## Further Reading
- OpenAI structured outputs: https://platform.openai.com/docs/guides/structured-outputs
- JSON RFC 8259: https://www.rfc-editor.org/rfc/rfc8259
- OpenClaw docs: https://docs.openclaw.ai
